# NB07 — Fork Choice & Reorgs

Extend NB04's chain and NB06's P2P node to support competing chains,
the longest-chain rule, and live reorg debugging.

**Dependencies:** `_lib/chain.py` (NB04), `_lib/peer.py` (NB06),
`_lib/rlp_min.py`, `_lib/keccak.py`, `_lib/ecdsa.py`.


## 1. What is a fork?

Two nodes can each build a different block at the same height (network
latency, validator disagreement, partition). Both look valid locally.
The protocol must pick a winner **deterministically**.

Pre-Merge Ethereum used the **longest-chain rule** (Nakamoto consensus):
the chain with the most blocks wins. Post-Merge Ethereum uses
**LMD-GHOST + Casper FFG**: validators vote on chain tips with weighted
attestations, and finality is checkpoint-based — more than two thirds of
stake must vote before a checkpoint is locked irreversibly (with slashing
for equivocators).

We implement the simple longest-chain rule and watch a **reorg** happen
live: one node discards its tip and adopts a competing (longer) chain.
The structural primitive — blocks forming a parent-hash tree, with a
selectable tip — is the same in both schemes; the consensus rule is just
a function over that tree.


## 2. Extended protocol — new message types

NB06 established:

| Type | Code | Payload |
|------|------|---------|
| HELLO | `0x01` | `node_id (8 bytes) \|\| port (2 bytes BE)` |
| ANNOUNCE_TX | `0x04` | `tx_hash (32 bytes)` |
| GET_TX | `0x05` | `tx_hash (32 bytes)` |
| TX | `0x06` | `tx_bytes (arbitrary)` |

NB07 adds four block-sync messages:

| Type | Code | Payload |
|------|------|---------|
| GET_HEAD | `0x10` | (none) |
| HEAD | `0x11` | `number(4 BE) \|\| hash(32)` |
| GET_BLOCK | `0x12` | `number(4 BE)` |
| BLOCK | `0x13` | `block_bytes (RLP-encoded)` |

These four messages form a minimal **sync sub-protocol**: ask a peer for
its best head, walk back to find the common ancestor, then download and
replay the peer's branch.


## 3. Block + Tx serialization (extend `_lib/chain.py`)

We need to ship blocks over TCP, so every `Block` and `Tx` must know how
to serialize itself to bytes and reconstruct from bytes.

We re-use the `rlp` package (already installed) for decoding — its
`rlp.decode` returns raw `bytes` lists that we can interpret field-by-field.
Our existing `rlp_encode` from `_lib/rlp_min.py` handles encoding.

We also add `replay_chain_from_genesis(blocks, genesis_state)` which is
required by the reorg path: when we rewind to a common ancestor we must
rebuild the state from scratch rather than un-apply transactions (which
would require inverse operations).


In [1]:
%%writefile _lib/chain.py
"""chain.py — minimal single-node blockchain primitives.

Provides Account, Tx, Block, Chain and the helpers get_account, apply_tx,
make_tx.  Imported by NB05+ notebooks.

NB07 adds: Tx.to_bytes / Tx.from_bytes, Block.to_bytes / Block.from_bytes,
and replay_chain_from_genesis.

Design notes:
- Block headers commit to transactions via concatenated tx hashes.
  NB08 replaces this with a Merkle root.
- No gas, no consensus, no networking.  Those come in NB07/NB05+.
"""

import copy
import time
from dataclasses import dataclass, field

from .rlp_min import rlp_encode
from .keccak import keccak256
from .ecdsa import sign_digest, recover_address, priv_to_address


# ---------------------------------------------------------------------------
# Account and State
# ---------------------------------------------------------------------------

@dataclass
class Account:
    balance: int = 0
    nonce: int = 0


State = dict  # addr (str) -> Account


def get_account(state: State, addr: str) -> Account:
    """Return account for *addr*, auto-creating a zero account if absent."""
    if addr not in state:
        state[addr] = Account()
    return state[addr]


# ---------------------------------------------------------------------------
# Transaction
# ---------------------------------------------------------------------------

@dataclass
class Tx:
    sender: str
    to: str
    value: int
    nonce: int
    r: bytes = field(default=b'')
    s: bytes = field(default=b'')
    y_parity: int = 0

    def unsigned_bytes(self) -> bytes:
        return rlp_encode([
            bytes.fromhex(self.sender.removeprefix('0x')),
            bytes.fromhex(self.to.removeprefix('0x')),
            self.value,
            self.nonce,
        ])

    def hash(self) -> bytes:
        return keccak256(self.unsigned_bytes() + self.r + self.s + bytes([self.y_parity]))

    def to_bytes(self) -> bytes:
        return rlp_encode([
            bytes.fromhex(self.sender.removeprefix('0x')),
            bytes.fromhex(self.to.removeprefix('0x')),
            self.value,
            self.nonce,
            self.r,
            self.s,
            self.y_parity,
        ])

    @classmethod
    def from_bytes(cls, data: bytes) -> 'Tx':
        import rlp as _rlp
        fields = _rlp.decode(data)
        sender = '0x' + fields[0].hex()
        to     = '0x' + fields[1].hex()
        value  = int.from_bytes(fields[2], 'big') if fields[2] else 0
        nonce  = int.from_bytes(fields[3], 'big') if fields[3] else 0
        r, s   = fields[4], fields[5]
        yp     = int.from_bytes(fields[6], 'big') if fields[6] else 0
        return cls(sender=sender, to=to, value=value, nonce=nonce, r=r, s=s, y_parity=yp)


def make_tx(priv: bytes, to: str, value: int, nonce: int) -> 'Tx':
    """Build and sign a Tx from a private key."""
    sender = priv_to_address(priv)
    t = Tx(sender, to, value, nonce)
    digest = keccak256(t.unsigned_bytes())
    t.r, t.s, t.y_parity = sign_digest(priv, digest)
    return t


# ---------------------------------------------------------------------------
# apply_tx
# ---------------------------------------------------------------------------

def apply_tx(state: State, tx: 'Tx') -> None:
    """Validate and apply *tx* to *state* (mutates in place).

    Raises AssertionError on any validation failure; caller is responsible
    for rolling back state if atomicity is required.
    """
    # 1. Verify signature recovers the claimed sender
    digest = keccak256(tx.unsigned_bytes())
    recovered = recover_address(digest, tx.r, tx.s, tx.y_parity)
    assert recovered == tx.sender, f'bad signature: recovered {recovered}, claims {tx.sender}'
    # 2. Nonce check
    sender = get_account(state, tx.sender)
    assert tx.nonce == sender.nonce, f'bad nonce: got {tx.nonce}, want {sender.nonce}'
    # 3. Balance check
    assert sender.balance >= tx.value, f'insufficient: have {sender.balance}, need {tx.value}'
    # 4. Apply
    sender.balance -= tx.value
    get_account(state, tx.to).balance += tx.value
    sender.nonce += 1


# ---------------------------------------------------------------------------
# Block
# ---------------------------------------------------------------------------

@dataclass
class Block:
    number: int
    parent_hash: bytes
    txs: list
    timestamp: int

    def header_bytes(self) -> bytes:
        tx_hashes_concat = b''.join(t.hash() for t in self.txs)
        return rlp_encode([
            self.number,
            self.parent_hash,
            self.timestamp,
            tx_hashes_concat,
        ])

    def hash(self) -> bytes:
        return keccak256(self.header_bytes())

    def to_bytes(self) -> bytes:
        tx_blobs = [t.to_bytes() for t in self.txs]
        return rlp_encode([
            self.number,
            self.parent_hash,
            self.timestamp,
            tx_blobs,
        ])

    @classmethod
    def from_bytes(cls, data: bytes) -> 'Block':
        import rlp as _rlp
        fields = _rlp.decode(data)
        number      = int.from_bytes(fields[0], 'big') if fields[0] else 0
        parent_hash = fields[1]
        timestamp   = int.from_bytes(fields[2], 'big') if fields[2] else 0
        txs = [Tx.from_bytes(b) for b in fields[3]]
        return cls(number=number, parent_hash=parent_hash, txs=txs, timestamp=timestamp)


# ---------------------------------------------------------------------------
# Chain
# ---------------------------------------------------------------------------

class Chain:
    """Single-node, in-process blockchain."""

    def __init__(self, genesis_state: State):
        self.state = genesis_state
        genesis = Block(0, b'\x00' * 32, [], int(time.time()))
        self.blocks = [genesis]

    @property
    def head(self) -> Block:
        return self.blocks[-1]

    def propose(self, txs: list) -> Block:
        """Apply *txs* atomically and append a new block.

        If any transaction fails validation the state is rolled back and the
        exception (AssertionError or ValueError) propagates to the caller.
        """
        snap = copy.deepcopy(self.state)
        try:
            for t in txs:
                apply_tx(self.state, t)
        except (AssertionError, ValueError):
            self.state = snap
            raise
        blk = Block(self.head.number + 1, self.head.hash(), txs, int(time.time()))
        self.blocks.append(blk)
        return blk


# ---------------------------------------------------------------------------
# replay_chain_from_genesis  (NB07)
# ---------------------------------------------------------------------------

def replay_chain_from_genesis(blocks, genesis_state) -> dict:
    """Reapply every tx in *blocks* to a fresh copy of *genesis_state*.

    blocks[0] is the genesis block (no txs, skipped).
    Returns the resulting state dict.

    This is used after a reorg: we rewound `chain.blocks` to the common
    ancestor so we must rebuild state from scratch rather than un-applying
    transactions (which would require inverse operations).
    """
    state = copy.deepcopy(genesis_state)
    for blk in blocks[1:]:
        for t in blk.txs:
            apply_tx(state, t)
    return state


Overwriting _lib/chain.py


## 4. Verify round-trip serialization

Build a block with a real signed transaction, serialize it, deserialize it,
and confirm that `Block.from_bytes(blk.to_bytes()).hash() == blk.hash()`.
This proves the wire format preserves all header fields exactly.


In [2]:
import importlib, sys, os

if '.' not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

import _lib.chain
importlib.reload(_lib.chain)
from _lib.chain import Account, Tx, Block, Chain, apply_tx, make_tx, get_account
from _lib.ecdsa import gen_private_key, priv_to_address

# Build a signed transaction
alice_priv = gen_private_key()
alice = priv_to_address(alice_priv)
bob_priv = gen_private_key()
bob = priv_to_address(bob_priv)

tx = make_tx(alice_priv, bob, 42, nonce=0)

# Verify Tx round-trip
tx2 = Tx.from_bytes(tx.to_bytes())
assert tx2.sender == tx.sender, f'sender mismatch: {tx2.sender!r} != {tx.sender!r}'
assert tx2.to     == tx.to
assert tx2.value  == tx.value
assert tx2.nonce  == tx.nonce
assert tx2.r      == tx.r
assert tx2.s      == tx.s
assert tx2.y_parity == tx.y_parity
print('Tx round-trip OK')

# Build a Block wrapping that tx
genesis = Block(0, b'\x00' * 32, [], 1_700_000_000)
blk = Block(1, genesis.hash(), [tx], 1_700_000_001)

# Verify Block round-trip
blk2 = Block.from_bytes(blk.to_bytes())
assert blk2.hash() == blk.hash(), (
    f'hash mismatch after round-trip:\n'
    f'  original  {blk.hash().hex()}\n'
    f'  roundtrip {blk2.hash().hex()}'
)
print(f'Block round-trip OK  hash={blk.hash().hex()[:16]}...')
print(f'  number={blk2.number}  txs={len(blk2.txs)}  parent={blk2.parent_hash.hex()[:8]}')


Tx round-trip OK
Block round-trip OK  hash=c93cf9e33461d092...
  number=1  txs=1  parent=b30d3f17


## 5. Extend `_lib/peer.py` — chain integration

Three changes to `Node`:

1. **Optional `chain` / `genesis_state` arguments** — so a node can hold
   a live chain and respond to block-sync messages.
2. **`_peer_loop` handles `GET_HEAD`, `GET_BLOCK`, and `BLOCK`** —
   the server side of the sync protocol.
3. **`_handle_block`, `try_sync`, `gossip_block` methods** —
   the client side: receive a pushed block, pull a peer's chain, broadcast.

**Socket isolation for sync requests:** `try_sync` opens a *fresh*
TCP connection to the peer rather than reusing the shared gossip socket.
This avoids a race between the background `_peer_loop` reader and the
synchronous request/response pattern that `try_sync` needs.


In [3]:
%%writefile _lib/peer.py
"""peer.py — toy Ethereum-style P2P node over TCP.

Message types
-------------
HELLO        0x01  -- node_id (8 bytes) || port (2 bytes BE)
ANNOUNCE_TX  0x04  -- tx_hash (32 bytes)
GET_TX       0x05  -- tx_hash (32 bytes)
TX           0x06  -- tx_bytes (arbitrary)
GET_HEAD     0x10  -- (none)
HEAD         0x11  -- number(4 BE) || hash(32)
GET_BLOCK    0x12  -- number(4 BE)
BLOCK        0x13  -- block_bytes (RLP)

Codes 0x02 (GETPEERS) and 0x03 (PEERS) are reserved for a future session.
"""

import copy
import math
import random
import secrets
import socket
import struct
import threading

from .framing import send_msg, recv_msg
from .keccak import keccak256

# ---------------------------------------------------------------------------
# Message type constants
# ---------------------------------------------------------------------------

HELLO       = 0x01
ANNOUNCE_TX = 0x04
GET_TX      = 0x05
TX          = 0x06
GET_HEAD    = 0x10
HEAD        = 0x11
GET_BLOCK   = 0x12
BLOCK       = 0x13


# ---------------------------------------------------------------------------
# Encode / decode helpers
# ---------------------------------------------------------------------------

def encode_msg(msg_type: int, body: bytes) -> bytes:
    """Prepend the single-byte type code to *body*."""
    return bytes([msg_type]) + body


def decode_msg(payload: bytes) -> tuple:
    """Split a received payload into (msg_type, body)."""
    return payload[0], payload[1:]


# ---------------------------------------------------------------------------
# Node class
# ---------------------------------------------------------------------------

class Node:
    """A dual-role (listener + dialer) P2P node.

    Each node:
    * Listens on *port* for inbound peers.
    * Can dial out to known peers via :meth:`connect`.
    * Gossips new transactions using sqrt-fanout (the eth/68 pattern).
    * Optionally holds a Chain; supports block-sync via try_sync.
    """

    def __init__(self, port: int, chain=None, genesis_state=None):
        self.port = port
        self.node_id = secrets.token_bytes(8)
        self.peers: dict = {}  # peer_id -> socket
        self.seen_tx: set = set()
        self.tx_pool: dict = {}  # hash -> raw bytes
        self.lock = threading.Lock()
        self._stop = threading.Event()
        self._listener_sock = None
        self.chain = chain
        self.genesis_state = copy.deepcopy(genesis_state) if genesis_state is not None else None

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    def start(self):
        """Start the background listener thread."""
        threading.Thread(target=self._listen, daemon=True).start()

    def stop(self):
        """Signal all threads to exit and close all sockets."""
        self._stop.set()
        with self.lock:
            for pid, conn in list(self.peers.items()):
                try:
                    conn.shutdown(socket.SHUT_RDWR)
                except OSError:
                    pass
                try:
                    conn.close()
                except OSError:
                    pass
            self.peers.clear()
        if self._listener_sock is not None:
            try:
                self._listener_sock.close()
            except OSError:
                pass

    # ------------------------------------------------------------------
    # Listener
    # ------------------------------------------------------------------

    def _listen(self):
        srv = socket.socket()
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(("127.0.0.1", self.port))
        srv.listen(20)
        srv.settimeout(0.5)  # allows _stop checks
        self._listener_sock = srv
        while not self._stop.is_set():
            try:
                conn, _ = srv.accept()
            except socket.timeout:
                continue
            except OSError:
                break
            threading.Thread(
                target=self._handshake_inbound,
                args=(conn,),
                daemon=True,
            ).start()

    # ------------------------------------------------------------------
    # Handshake -- inbound direction
    # ------------------------------------------------------------------

    def _handshake_inbound(self, conn):
        try:
            conn.settimeout(2.0)
            payload = recv_msg(conn)
            mt, body = decode_msg(payload)
            if mt != HELLO:
                conn.close()
                return
            peer_id = body[:8]
            peer_port = struct.unpack(">H", body[8:10])[0]
            with self.lock:
                if peer_id == self.node_id or peer_id in self.peers:
                    conn.close()
                    return
                self.peers[peer_id] = conn
            send_msg(conn, encode_msg(HELLO, self.node_id + struct.pack(">H", self.port)))
            print(f"[{self.port}] inbound peer {peer_id.hex()} from :{peer_port}")
            self._peer_loop(peer_id, conn)
        except Exception:
            conn.close()

    # ------------------------------------------------------------------
    # Handshake -- outbound direction
    # ------------------------------------------------------------------

    def connect(self, host: str, port: int) -> bool:
        """Dial *host:port*, exchange HELLO, and start a peer loop."""
        try:
            conn = socket.socket()
            conn.settimeout(2.0)
            conn.connect((host, port))
            send_msg(conn, encode_msg(HELLO, self.node_id + struct.pack(">H", self.port)))
            payload = recv_msg(conn)
            mt, body = decode_msg(payload)
            if mt != HELLO:
                conn.close()
                return False
            peer_id = body[:8]
            with self.lock:
                if peer_id == self.node_id or peer_id in self.peers:
                    conn.close()
                    return False
                self.peers[peer_id] = conn
            print(f"[{self.port}] outbound peer {peer_id.hex()} -> :{port}")
            threading.Thread(
                target=self._peer_loop,
                args=(peer_id, conn),
                daemon=True,
            ).start()
            return True
        except (OSError, ConnectionError):
            return False

    # ------------------------------------------------------------------
    # Per-peer message loop
    # ------------------------------------------------------------------

    def _peer_loop(self, peer_id: bytes, conn: socket.socket):
        try:
            while not self._stop.is_set():
                try:
                    payload = recv_msg(conn)
                except socket.timeout:
                    continue
                mt, body = decode_msg(payload)
                if mt == ANNOUNCE_TX:
                    self._handle_announce(peer_id, body)
                elif mt == GET_TX:
                    with self.lock:
                        tx_body = self.tx_pool.get(body)
                    if tx_body is not None:
                        try:
                            send_msg(conn, encode_msg(TX, tx_body))
                        except OSError:
                            pass
                elif mt == TX:
                    self._handle_tx(body)
                elif mt == GET_HEAD:
                    if self.chain is not None:
                        with self.lock:
                            head = self.chain.head
                        try:
                            send_msg(conn, encode_msg(HEAD,
                                struct.pack(">I", head.number) + head.hash()))
                        except OSError:
                            pass
                elif mt == GET_BLOCK:
                    n = struct.unpack(">I", body[:4])[0]
                    with self.lock:
                        if self.chain is not None and 0 <= n < len(self.chain.blocks):
                            blk_bytes = self.chain.blocks[n].to_bytes()
                        else:
                            blk_bytes = None
                    if blk_bytes is not None:
                        try:
                            send_msg(conn, encode_msg(BLOCK, blk_bytes))
                        except OSError:
                            pass
                elif mt == BLOCK:
                    self._handle_block(body)
        except (ConnectionError, OSError, ValueError):
            pass
        finally:
            with self.lock:
                self.peers.pop(peer_id, None)

    # ------------------------------------------------------------------
    # Message handlers
    # ------------------------------------------------------------------

    def _handle_announce(self, peer_id: bytes, tx_hash: bytes):
        with self.lock:
            if tx_hash in self.seen_tx:
                return
            conn = self.peers.get(peer_id)
        if conn is not None:
            try:
                send_msg(conn, encode_msg(GET_TX, tx_hash))
            except OSError:
                pass

    def _handle_tx(self, tx_bytes: bytes):
        h = keccak256(tx_bytes)
        with self.lock:
            if h in self.seen_tx:
                return
            self.seen_tx.add(h)
            self.tx_pool[h] = tx_bytes
        print(f"[{self.port}] got tx {h.hex()[:8]}...")
        self._gossip(h)

    def _handle_block(self, blk_bytes: bytes):
        """Called when an unsolicited BLOCK arrives -- extend chain if it fits on head."""
        from .chain import Block, apply_tx
        blk = Block.from_bytes(blk_bytes)
        with self.lock:
            if self.chain is None:
                return
            if blk.number != self.chain.head.number + 1:
                return
            if blk.parent_hash != self.chain.head.hash():
                return
            snap = copy.deepcopy(self.chain.state)
            try:
                for t in blk.txs:
                    apply_tx(self.chain.state, t)
                self.chain.blocks.append(blk)
                print(f"[{self.port}] extended chain to height {blk.number}")
            except (AssertionError, ValueError) as e:
                self.chain.state = snap
                print(f"[{self.port}] rejected block {blk.number}: {e}")

    # ------------------------------------------------------------------
    # Block sync
    # ------------------------------------------------------------------

    def try_sync(self, peer_port: int) -> bool:
        """Open a fresh connection to *peer_port* and sync if the peer is longer.

        Uses a dedicated short-lived socket so the sync request/response
        sequence does not race with the background _peer_loop reader.
        Returns True if a reorg was performed, False otherwise.
        """
        from .chain import Block, apply_tx, replay_chain_from_genesis
        if self.chain is None:
            return False

        # Open a fresh socket just for sync
        try:
            sock = socket.socket()
            sock.settimeout(3.0)
            sock.connect(('127.0.0.1', peer_port))
            # Use a fresh ephemeral node_id so the server does not reject
            # this socket as a duplicate of our existing gossip connection.
            ephemeral_id = secrets.token_bytes(8)
            send_msg(sock, encode_msg(HELLO, ephemeral_id + struct.pack(">H", self.port)))
            hello_payload = recv_msg(sock)
            mt, _ = decode_msg(hello_payload)
            if mt != HELLO:
                sock.close()
                return False
        except (OSError, ConnectionError, socket.timeout):
            return False

        try:
            # Ask for peer's best head
            send_msg(sock, encode_msg(GET_HEAD, b''))
            payload = recv_msg(sock)
            mt, body = decode_msg(payload)
            if mt != HEAD:
                return False
            peer_num = struct.unpack(">I", body[:4])[0]

            with self.lock:
                my_head_num = self.chain.head.number
            if peer_num <= my_head_num:
                return False

            # Walk back from peer's head until we find a common ancestor
            needed = []  # reversed: peer's head first
            cur_num = peer_num
            while True:
                send_msg(sock, encode_msg(GET_BLOCK, struct.pack(">I", cur_num)))
                payload = recv_msg(sock)
                mt, body = decode_msg(payload)
                if mt != BLOCK:
                    return False
                blk = Block.from_bytes(body)
                needed.append(blk)
                with self.lock:
                    local_len = len(self.chain.blocks)
                    if cur_num < local_len and self.chain.blocks[cur_num].hash() == blk.hash():
                        # Found common ancestor
                        break
                if cur_num == 0:
                    break
                cur_num -= 1

            # needed[-1] is the common-ancestor block at height cur_num
            # new_blocks is everything above the common ancestor, forward order
            common_num = needed[-1].number
            new_blocks = list(reversed(needed[:-1]))

            with self.lock:
                rewound = self.chain.blocks[common_num + 1:]
                self.chain.blocks = self.chain.blocks[:common_num + 1]
                # Rebuild state from genesis through the common ancestor
                self.chain.state = replay_chain_from_genesis(
                    self.chain.blocks, self.genesis_state
                )
                # Apply peer's branch
                for b in new_blocks:
                    for t in b.txs:
                        apply_tx(self.chain.state, t)
                    self.chain.blocks.append(b)
                print(f"[{self.port}] REORG: dropped {len(rewound)} block(s), applied {len(new_blocks)}")
            return True
        except (OSError, ConnectionError, socket.timeout):
            return False
        finally:
            try:
                sock.close()
            except OSError:
                pass

    def gossip_block(self, blk):
        """Broadcast a new block to all peers."""
        blk_bytes = blk.to_bytes()
        with self.lock:
            peers = list(self.peers.values())
        for conn in peers:
            try:
                send_msg(conn, encode_msg(BLOCK, blk_bytes))
            except OSError:
                pass

    # ------------------------------------------------------------------
    # Sqrt-fanout gossip
    # ------------------------------------------------------------------

    def _gossip(self, tx_hash: bytes):
        """Send TX body to sqrt(N) peers; ANNOUNCE_TX (hash only) to the rest."""
        with self.lock:
            peers = list(self.peers.items())
            body = self.tx_pool.get(tx_hash)
        if not peers or body is None:
            return
        k = max(1, int(math.sqrt(len(peers))))
        full_recipients = set(p[0] for p in random.sample(peers, min(k, len(peers))))
        for pid, conn in peers:
            try:
                if pid in full_recipients:
                    send_msg(conn, encode_msg(TX, body))
                else:
                    send_msg(conn, encode_msg(ANNOUNCE_TX, tx_hash))
            except OSError:
                pass

    def submit_tx(self, tx_bytes: bytes):
        """Inject a new transaction into this node and gossip it."""
        h = keccak256(tx_bytes)
        with self.lock:
            self.seen_tx.add(h)
            self.tx_pool[h] = tx_bytes
        self._gossip(h)


Overwriting _lib/peer.py


## 6. Demo setup

Two helpers: `fresh_chain()` returns a new chain seeded from the same
genesis state, and `fresh_node(port)` bundles chain + node together.
Both alice and bob are deterministic so every run starts from the same
initial balances.


In [4]:
import importlib, sys, os, copy, time

if '.' not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

import _lib.chain, _lib.peer
importlib.reload(_lib.chain)
importlib.reload(_lib.peer)
from _lib.chain import Chain, Account, make_tx
from _lib.peer import Node
from _lib.ecdsa import gen_private_key, priv_to_address

alice_priv = gen_private_key(); alice = priv_to_address(alice_priv)
bob_priv   = gen_private_key(); bob   = priv_to_address(bob_priv)
print('alice:', alice)
print('bob:  ', bob)

GENESIS_STATE = {alice: Account(balance=10_000, nonce=0)}

def fresh_chain():
    return Chain(copy.deepcopy(GENESIS_STATE))

def fresh_node(port):
    return Node(port, chain=fresh_chain(), genesis_state=GENESIS_STATE)


alice: 0xe710203aea24dd113fdbaa6e7cee6d819b238a6c
bob:   0x1e5d7fd7bcf6640f056b0241a4911e681b675d14


## 7. Naive sync: two nodes, no fork

Node 1 proposes a block. Node 2 starts empty (only genesis). When they
connect, we call `n2.try_sync(n1.port)` — node 2 detects that node 1 is
one block ahead, fetches it, and both heads converge.


In [5]:
n1 = fresh_node(5601); n2 = fresh_node(5602)
n1.start(); n2.start()
time.sleep(0.2)

tx_a = make_tx(alice_priv, bob, 100, nonce=0)
blk1 = n1.chain.propose([tx_a])
print(f'node 1 head: #{n1.chain.head.number} {n1.chain.head.hash().hex()[:8]}')

# Connect the nodes (gossip channel -- separate from the sync socket)
n1.connect('127.0.0.1', n2.port)
time.sleep(0.2)

# Explicit sync: node 2 asks node 1 for its head via a fresh socket
n2.try_sync(n1.port)
print(f'node 2 head: #{n2.chain.head.number} {n2.chain.head.hash().hex()[:8]}')

assert n1.chain.head.hash() == n2.chain.head.hash(), (
    f'heads differ: n1={n1.chain.head.hash().hex()[:8]} n2={n2.chain.head.hash().hex()[:8]}'
)
print('nodes agree on head')

n1.stop(); n2.stop()


node 1 head: #1 e7ec7a94
[5602] inbound peer 4fac2a1ca1d01072 from :5601
[5601] outbound peer 293e3c813a2dd25e -> :5602


[5601] inbound peer 9b63a2ccba9bf403 from :5602
[5602] REORG: dropped 0 block(s), applied 1
node 2 head: #1 e7ec7a94
nodes agree on head


## 8. `dump_chain` helper

A quick ASCII view of a node's chain, used to make the reorg visible.


In [6]:
def dump_chain(label, node):
    print(f'\n{label} (head = #{node.chain.head.number}):')
    for b in node.chain.blocks:
        print(f'  #{b.number}  hash={b.hash().hex()[:8]}'
              f'  parent={b.parent_hash.hex()[:8]}  txs={len(b.txs)}')


## 9. Cause a fork

Both nodes are isolated (not yet connected). Each proposes its own block
at height 1, but with **different transactions**, so the blocks have
different hashes — a real fork.

Then node 2 extends its chain to height 2 with a second block.
At this point:

* Node 1: height 1 (fork-A tip)
* Node 2: height 2 (fork-B tip, longer)


In [7]:
n1 = fresh_node(5611); n2 = fresh_node(5612)
n1.start(); n2.start()
time.sleep(0.2)

# Each node proposes its own block at height 1 with different amounts
tx1 = make_tx(alice_priv, bob, 100, nonce=0)  # node 1's version
tx2 = make_tx(alice_priv, bob, 200, nonce=0)  # node 2's version (different value)
n1.chain.propose([tx1])
n2.chain.propose([tx2])

dump_chain('BEFORE CONNECT -- node 1', n1)
dump_chain('BEFORE CONNECT -- node 2', n2)

# Make node 2 longer by adding a second block
# Node 2's alice nonce is now 1 (she spent in the first block)
tx3 = make_tx(alice_priv, bob, 50, nonce=1)
n2.chain.propose([tx3])

dump_chain('AFTER node 2 extends -- node 2', n2)
print(f'\nnode 1 chain length: {len(n1.chain.blocks)}')
print(f'node 2 chain length: {len(n2.chain.blocks)}')
assert n1.chain.head.hash() != n2.chain.head.hash(), 'expected different heads'
print('fork confirmed: heads differ')



BEFORE CONNECT -- node 1 (head = #1):
  #0  hash=0c1c1e2d  parent=00000000  txs=0
  #1  hash=e7ec7a94  parent=0c1c1e2d  txs=1

BEFORE CONNECT -- node 2 (head = #1):
  #0  hash=0c1c1e2d  parent=00000000  txs=0
  #1  hash=81f1311f  parent=0c1c1e2d  txs=1

AFTER node 2 extends -- node 2 (head = #2):
  #0  hash=0c1c1e2d  parent=00000000  txs=0
  #1  hash=81f1311f  parent=0c1c1e2d  txs=1
  #2  hash=e8e984ea  parent=81f1311f  txs=1

node 1 chain length: 2
node 2 chain length: 3
fork confirmed: heads differ


## 10. Connect and trigger the reorg

Node 1 calls `try_sync(n2.port)`. It detects node 2 is ahead by one
block, walks back to find the common ancestor (genesis), unwinds its own
fork-A tip, replays state from genesis through node 2's two blocks, and
adopts node 2's head. This is a **reorg** — node 1's previously-canonical
block is now orphaned.


In [8]:
print('--- node 1 syncing from node 2 ---')
result = n1.try_sync(n2.port)
print(f'try_sync returned: {result}')

dump_chain('AFTER REORG -- node 1', n1)
dump_chain('AFTER REORG -- node 2', n2)

assert n1.chain.head.hash() == n2.chain.head.hash(), (
    f'reorg failed: n1={n1.chain.head.hash().hex()[:8]} n2={n2.chain.head.hash().hex()[:8]}'
)
print('\nfork resolved by longest-chain rule')

# State consistency: alice should have 10_000 - 200 - 50 = 9_750
from _lib.chain import get_account
alice_bal = get_account(n1.chain.state, alice).balance
assert alice_bal == 9_750, f'expected alice 9750 after reorg, got {alice_bal}'
print(f'alice balance after reorg: {alice_bal} (correct)')

n1.stop(); n2.stop()


--- node 1 syncing from node 2 ---
[5612] inbound peer 4241918393591cd5 from :5611
[5611] REORG: dropped 1 block(s), applied 2
try_sync returned: True

AFTER REORG -- node 1 (head = #2):
  #0  hash=0c1c1e2d  parent=00000000  txs=0
  #1  hash=81f1311f  parent=0c1c1e2d  txs=1
  #2  hash=e8e984ea  parent=81f1311f  txs=1

AFTER REORG -- node 2 (head = #2):
  #0  hash=0c1c1e2d  parent=00000000  txs=0
  #1  hash=81f1311f  parent=0c1c1e2d  txs=1
  #2  hash=e8e984ea  parent=81f1311f  txs=1

fork resolved by longest-chain rule
alice balance after reorg: 9750 (correct)


## 11. Debugging exercise — invalid-block defense

What happens when a malicious peer sends a block with a tampered
transaction signature? We flip one byte in the `r` field and call
`_handle_block` directly. The node must **reject** the block and keep
its head at genesis.

This is the same defense used by real Ethereum clients: every transaction
is signature-checked before the block is accepted.


In [9]:
import time as _t
from _lib.chain import Block

n3 = fresh_node(5621)
n3.start()
time.sleep(0.1)

# Build a well-formed tx, then tamper with it BEFORE wrapping in a block
good_tx = make_tx(alice_priv, bob, 10, nonce=0)
good_tx.r = bytes([good_tx.r[0] ^ 0x01]) + good_tx.r[1:]  # flip one bit in r

evil_blk = Block(
    number=1,
    parent_hash=n3.chain.head.hash(),
    txs=[good_tx],
    timestamp=int(_t.time()),
)

n3._handle_block(evil_blk.to_bytes())
print(f'node 3 head: #{n3.chain.head.number} (expected 0 -- tampered block rejected)')
assert n3.chain.head.number == 0, (
    f'node accepted a tampered block! head is now #{n3.chain.head.number}'
)
print('tampered-block defense confirmed')

n3.stop()


[5621] rejected block 1: bad signature: recovered 0x749ee7a1422783ef16a355b90db336e574d88249, claims 0xe710203aea24dd113fdbaa6e7cee6d819b238a6c
node 3 head: #0 (expected 0 -- tampered block rejected)
tampered-block defense confirmed


## 12. Why real Ethereum is harder

We implemented the simplest possible fork-choice rule: most blocks wins.
Real Ethereum post-Merge uses **LMD-GHOST** (Latest Message Driven,
Greedy Heaviest Observed Sub-Tree) combined with **Casper FFG** finality:

* **LMD-GHOST**: at each fork, pick the sub-tree with the most validator
  vote weight, not merely the most blocks. This makes it harder to attack
  with a minority of stake.
* **Casper FFG**: every 32 slots a *checkpoint* is eligible for
  finalization. Once more than two thirds of stake has voted for a
  checkpoint's *source* and *target*, the checkpoint is **justified**;
  a subsequent supermajority finalizes it. Finalized blocks can never be
  reorged without *slashing* the violating validators (burning their stake).

What we have built is the **structural spine** used by both schemes:

* Blocks form a parent-hash tree (`parent_hash` field).
* State is derived by replaying transactions from genesis.
* A *fork-choice function* selects the canonical tip.
* A *reorg* rewinds to a common ancestor and replays the winning branch.

Swapping in LMD-GHOST means replacing the `peer_num > my_head_num` check
in `try_sync` with a weighted-attestation comparison — everything else
stays the same.
